# Figure 3 — a moving companion

**In the paper:** Fig. 3; Sect. 4.1 (Eq. 14); Sect. 4.2 (Eqs. 15–17); Sect. 4.3 (Eq. 18); the numbers in Sect. 4.5.

Fig. 2's chain on a companion that moves relative to its primary: the detections are taken back to
one reference epoch before they are pooled, and the fit has four parameters (five with a parallax
difference).

| step | task | calls | paper |
|---|---|---|---|
| 1 | the scan law and the moving scene | `sim.Scene.pair`, `scanlaw.load_scan_law`, `scanlaw.load_parallax_factor` | Sect. 4.1, Eq. 14; Sect. 4.2, Eq. 16 |
| 2 | the windows of case (a) | `sim.simulate_window`, `ins.project_al` | — |
| 3 | the per-window reduction | `red.ladder_window`, `red.delivery_mask`, `red.sigma_eta`; in one call `sim.harvest_campaign` | Sect. 2.2 |
| 4 | the detection-set fit: the sinogram of a moving source | `dyn.motion_design`, `dyn.fit_motion` | Sect. 4.2, Eqs. 15, 16 |
| 5 | the seed: the map detects, a circle selects, the sinogram fit refines and cleans; the compensation to the reference epoch | `bp.map_peak`, `bp.peak_consistency`, `dyn.fit_motion_robust`, `dyn.fit_motion`, `dyn.to_reference_epoch`; in one call `dyn.clean_detection_set` | Sect. 4.2, Eq. 17 |
| 6 | the window-level likelihood | `joint.scene_positions_al`, `joint.stacked_design`, `joint.loglike_profiled`, `joint.loglike_windows` | Sect. 4.3, Eq. 18 |
| 7 | the fit | `opt.minimize_restarts`, `joint.joint_fisher`, `joint.joint_covariance`; in one call `joint.fit_joint` | Sect. 4.3 |
| 8 | the compensated sinogram, the top row of the figure | `dyn.to_reference_epoch`, `ins.project_al` | Fig. 3, top row |
| 9 | the three runs: (a) bound, its null test with a parallax difference, (b) chance alignment | the notebook's `chain` (steps 2–8) | Sect. 4.5 |
| 10 | the figure | `bp.backprojection_map` | Fig. 3 |
| 11 | over 300 draws: the detection-set fit is biased, the window-level fit is not | — (reads `stored/fig3_coverage.npz`) | Sect. 4.5 |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**The scenes** (Sect. 4.5). 200 mas at PA 110° at the reference epoch, 𝒮 = 0.3, the 47 transits of
Fig. 2 with their epochs and parallax factors. (a) A bound companion, relative proper motion
(3, 9) mas/yr, 48 mas of drift over the 5.1-yr span. (b) A chance-aligned star 2 kpc behind a
primary at 120 pc, proper motion (4, −3) mas/yr, Δϖ = −7.83 mas. Seed 14.

**Paper and code.** The detection-set fit = the probe (`dyn.fit_motion`); Eq. 17 = `dyn.to_reference_epoch`;
$f_\varpi$ = `fpar`; 𝒮 = `Q`. Each one-call form (`sim.harvest_campaign`, `dyn.clean_detection_set`,
`joint.fit_joint`) is asserted equal to the stage written out.

**What it reproduces** (Sect. 4.5). (a) The joint fit's proper motion (3.0 ± 0.6, 8.9 ± 0.4) mas/yr
(3.05 ± 0.56, 8.89 ± 0.37), 𝒮 = 0.302; the null test Δϖ = −0.1 ± 1.0 mas. (b) Δϖ = −7.4 ± 1.0 mas
(−7.45 ± 0.95). Over 300 draws: the detection-set fit biased by about 1 mas and 0.4–0.5 mas/yr, short
of the drift; the window-level fit unbiased and 2–5 times tighter. About 5 s; §11 reads
`stored/fig3_coverage.npz`.

In [ ]:
# the imports: numpy, matplotlib, and the scanlib modules; every scanlib call below carries its module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import reduction as red       # one window, or a campaign, reduced to detections
from scanlib import backprojection as bp   # the triangulation map and its peak
from scanlib import dynamics as dyn        # the fits to the detections (the sinogram fits)
from scanlib import joint                  # the joint fit of all windows, and its helpers
from scanlib import optimize as opt        # the optimizer of the joint fit

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in auxil/sim_margins_lambda_w)
FWHM_MAS = 100.0                           # the full width at half maximum of the LSF [mas]
PITCH_MAS = 59.0                           # the width of one sample along the scan [mas]
N_SAMPLES = 18                             # the number of samples of one window
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

SAVE_PDF = False                           # True: write pdf/fig3_moving_companion.pdf

## 1. The scan law and the moving scene (Sect. 4.1, Eq. 14; Sect. 4.2, Eq. 16)

Each transit has an epoch, τ = t − t₀ years from the reference epoch (the middle of the span), and
an along-scan parallax factor $f_\varpi$. The companion moves linearly (Eq. 14), so in window $j$

$$\eta_j = \alpha_0\sin\psi_j + \delta_0\cos\psi_j + \mu_{\alpha^*}\tau_j\sin\psi_j + \mu_\delta\tau_j\cos\psi_j + \Delta\varpi\,f_{\varpi,j}. \qquad (16)$$

The primary stays at η = 0 (the frame moves with it). Each window is reduced to two sources at most.

In [ ]:
# section 1: the two scenes, (a) bound and (b) a chance alignment, and the companion's truth at the reference
# epoch
RHO_MAS = 200.0                            # the companion's separation at the reference epoch [mas]
PA_DEG = 110.0                             # its position angle at the reference epoch [deg]
Q = 0.3                                    # its flux ratio, companion over primary
MU_A_BOUND = 3.0                           # (a) bound: relative proper motion in alpha* [mas/yr]
MU_D_BOUND = 9.0                           # (a) bound: relative proper motion in delta [mas/yr]
MU_A_INTERLOPER = 4.0                      # (b) interloper: relative proper motion in alpha* [mas/yr]
MU_D_INTERLOPER = -3.0                     # (b) interloper: relative proper motion in delta [mas/yr]
DPI_INTERLOPER = 1e3 / 2000.0 - 1e3 / 120.0   # (b) 1/(2 kpc) - 1/(120 pc) in mas: -7.83 mas
SEED = 14                                  # this figure's noise realization
K_MAX = 2                                  # at most two sources per window

# sim.Scene.pair: the primary at the origin and one companion at (rho, PA), flux ratio Q, with its relative motion
scene_a = sim.Scene.pair(RHO_MAS, PA_DEG, Q, pm=(MU_A_BOUND, MU_D_BOUND), dpi=0.0)
scene_b = sim.Scene.pair(RHO_MAS, PA_DEG, Q, pm=(MU_A_INTERLOPER, MU_D_INTERLOPER), dpi=DPI_INTERLOPER)
# positions has one row (dra*, ddec) per source; row 1 is the companion (row 0 is the primary). alpha0_true and
# delta0_true are alpha_0 and delta_0 of Eq. 16: the companion's true offset at the reference epoch [mas]
alpha0_true, delta0_true = scene_a.positions[1]
# the true parameter vectors, in the order of the fits: (alpha_0, delta_0, mu_alpha*, mu_delta[, d_parallax])
TRUTH_A = np.array([alpha0_true, delta0_true, MU_A_BOUND, MU_D_BOUND])
TRUTH_B = np.array([alpha0_true, delta0_true, MU_A_INTERLOPER, MU_D_INTERLOPER, DPI_INTERLOPER])

In [ ]:
# the scan law of the Pleiades centre: each transit's scan angle, epoch and along-scan parallax factor
psi, t = scanlaw.load_scan_law(scanlaw.PLEIADES)          # psi: scan angles [rad]; t: years from the reference epoch
fpar = scanlaw.load_parallax_factor(scanlaw.PLEIADES)      # f_varpi of each transit
law = (psi, t, fpar)                                       # the three arrays together, for sim.harvest_campaign
# Eq. 16 by hand for case (b) against the scene's own projection
eta_eq16 = ((alpha0_true + MU_A_INTERLOPER * t) * np.sin(psi) + (delta0_true + MU_D_INTERLOPER * t) * np.cos(psi)
            + DPI_INTERLOPER * fpar)
# the scene's own projection: along_scan returns the eta of every source in one transit, and [1] keeps the
# companion's (index 0 is the primary)
eta_from_scene = []
for j in range(psi.size):                                  # j counts the transits
    eta_of_sources = scene_b.along_scan(psi[j], t[j], fpar[j])
    eta_from_scene.append(eta_of_sources[1])
assert np.allclose(eta_eq16, eta_from_scene, rtol=0, atol=1e-9)

In [ ]:
# the scan law and the two scenes printed, with each scene's drift over the span
span = t.max() - t.min()                                   # from the first transit to the last [yr]
print(f"transits                        {psi.size}")
print(f"epochs from the reference [yr]  {t.min():+.2f} to {t.max():+.2f}  (span {span:.1f} yr)")
print(f"along-scan parallax factor      {fpar.min():+.2f} to {fpar.max():+.2f}")
for tag, truth in (("(a) bound", TRUTH_A), ("(b) interloper", TRUTH_B)):
    proper_motion = truth[2:4]                             # the third and fourth entries: (mu_a, mu_d) [mas/yr]
    drift = np.hypot(*proper_motion) * span                # the size of the proper motion times the span [mas]
    line = (f"{tag:15s} t0 offset ({truth[0]:.1f}, {truth[1]:.1f}) mas, "
            f"PM ({truth[2]:+.1f}, {truth[3]:+.1f}) mas/yr, drift {drift:.0f} mas")
    # truth[4] is the parallax difference, which only case (b) has (five entries)
    if truth.size == 5:
        line = line + f", d_parallax {truth[4]:+.2f} mas"
    print(line)

## 2. The windows of case (a)

As in Fig. 2, with each source at its position at that window's epoch.

In [ ]:
# section 2: render, the function that makes a campaign's windows, each source at its position at the
# window's epoch
SNR_PRIMARY = 100.0                        # the signal-to-noise ratio of the isolated primary (total flux over sigma)


def render(scene, seed=SEED):
    """The noisy windows of one scene over the scan law, one window per transit.

    Inputs
        scene   a sim.Scene: the positions, fluxes and relative motions of the sources, the primary first
        seed    the seed of the noise; the default is this figure's realization
    Returns three things
        windows       a list with one array of N_SAMPLES samples per transit, in the law's order
        sigma_noise   the noise sigma of every sample [flux per sample], the same for all windows
        eta_true      the true along-scan position [mas] of every source in every transit: one row per transit,
                      column 0 the primary, column 1 the companion"""
    # step 1, the random generator of the noise. The seed is one number; nothing else enters it, so two scenes
    # rendered with the same seed get the same noise draws
    rng_noise = np.random.default_rng(seed)

    # step 2, the noise sigma: S/N 100 for the isolated primary
    sigma_noise = sim.PRIMARY_FLUX / SNR_PRIMARY

    # step 3, one window per transit. j counts the transits
    windows = []
    eta_true = []
    for j in range(psi.size):
        eta_j = scene.along_scan(psi[j], t[j], fpar[j])                # Eq. 16 for every source at epoch t_j
        # one draw of noise per window, in the law's order
        samples = sim.simulate_window(eta_j, scene.fluxes, inst, rng_noise, sigma_noise)
        windows.append(samples)
        eta_true.append(eta_j)
    return windows, sigma_noise, np.array(eta_true)

In [ ]:
# truth: the windows of case (a), and how far the motion moves the companion from where a static one would sit
windows_a, sigma_noise, eta_true_a = render(scene_a)
print(f"windows {len(windows_a)} x {windows_a[0].size} samples, noise sigma {sigma_noise:.3f} per sample")
# ins.project_al: Eq. 9, the along-scan position of a fixed sky offset; here the companion if it did not move
eta_static = ins.project_al(alpha0_true, delta0_true, psi)
# eta_true_a[:, 1] keeps every window and the second column, the companion (column 0 is the primary)
eta_companion_a = eta_true_a[:, 1]
print(f"the motion moves the companion along the scan by up to "
      f"{np.abs(eta_companion_a - eta_static).max():.0f} mas from where a static one would sit")

## 3. The per-window reduction (Sect. 2.2)

Fig. 2's reduction; each detection also carries its window's epoch and parallax factor.

In [ ]:
# section 3: each window reduced on its own (as in Figs. 1 and 2); every detection keeps its window's epoch and
# parallax factor, which the motion fit needs


def reduce_windows(windows, sigma_noise):
    """The detections of a campaign: every window reduced on its own, the primary's own source left out.

    Inputs
        windows       the list of windows, one array of samples per transit (from render)
        sigma_noise   the noise sigma of every sample [flux per sample]
    Returns a table of the detections, a dictionary of six arrays with one entry per detection
        "window"   the number of the detection's window (its place in the scan law)
        "psi"      the scan angle of that window [rad]
        "t"        its epoch [yr from the reference epoch]
        "fpar"     its along-scan parallax factor
        "eta"      the detected along-scan position [mas]
        "sigma"    the error of that position [mas]
    A window with one source gives no detection (it is censored)."""
    # step 1, reduce each window and collect its detections. j counts the windows
    window_of_detection = []
    eta_of_detection = []
    sigma_of_detection = []
    for j, samples in enumerate(windows):
        # red.ladder_window: Fig. 1's reduction of one window. It returns the fitted positions [mas], the fitted
        # fluxes, the number of sources, and a dictionary of diagnostics (dropped here, the name _)
        eta, flux, k, _ = red.ladder_window(samples, inst, sigma_noise, k_max=K_MAX)
        # red.delivery_mask: True for every source but the one nearest the origin (the primary)
        delivered = red.delivery_mask(eta)
        if delivered.any():
            # red.sigma_eta: the Fisher error of each position (Eq. 8)
            sigma_eta = red.sigma_eta(eta, flux, inst, sigma_noise)
            for eta_one, sigma_one in zip(eta[delivered], sigma_eta[delivered]):
                window_of_detection.append(j)
                eta_of_detection.append(eta_one)
                sigma_of_detection.append(sigma_one)

    # step 2, the lists as arrays; an error that is not finite (a singular Fisher matrix) gets the package's
    # fallback value
    window = np.array(window_of_detection)
    eta_detected = np.array(eta_of_detection)
    sigma_detected = np.array(sigma_of_detection)
    sigma_detected = np.where(np.isfinite(sigma_detected), sigma_detected, red.SIGMA_DET_FALLBACK_MAS)

    # step 3, the table: psi[window] is the scan angle of each detection's window, and so on
    return dict(window=window, psi=psi[window], t=t[window], fpar=fpar[window], eta=eta_detected,
                sigma=sigma_detected)

In [ ]:
# the detections of case (a), and the number of censored windows
det_a = reduce_windows(windows_a, sigma_noise)
# np.unique keeps each window's number once, so its size is the number of windows that gave a detection
n_windows_detected_a = np.unique(det_a["window"]).size
print(f"(a): {det_a['eta'].size} detections from {n_windows_detected_a} windows; "
      f"{psi.size - n_windows_detected_a} windows censored")

In [ ]:
# the convenience: render and reduce in one call, from the same seed
# the generator has render's seed, so sim.harvest_campaign draws the same noise as render did
rng_harvest = np.random.default_rng(SEED)
harvest = sim.harvest_campaign(scene_a, law, rng_harvest, inst, sigma_noise, k_max=K_MAX, return_windows=True)
# the windows are equal one by one (a pairs with b), and the noise sigma is the same
assert all(np.array_equal(a, b) for a, b in zip(harvest.windows, windows_a)) and harvest.sigma_noise == sigma_noise
# each array of the harvest (a) against the same array of the table written out above (b)
for a, b in ((harvest.psi, det_a["psi"]), (harvest.tau, det_a["t"]), (harvest.fpar, det_a["fpar"]),
             (harvest.eta, det_a["eta"]), (harvest.sigma, det_a["sigma"])):
    assert np.array_equal(a, b)
print("sim.harvest_campaign reproduces the windows and the detections exactly")

## 4. The detection-set fit (Sect. 4.2, Eqs. 15, 16)

The detections are still linear in the motion: Eq. 15 adds the columns τ sin ψ and τ cos ψ to the
static design of Eq. 11 (and $f_\varpi$ for Eq. 16). One weighted least-squares solve,
$\hat\theta = (\mathbf X^{\mathsf T}\mathbf W\mathbf X)^{-1}\mathbf X^{\mathsf T}\mathbf W\hat{\vec\eta}$, $\mathbf W = {\rm diag}(\sigma_{\eta,i}^{-2})$,
gives the reference-epoch offset and the proper motion: the **probe**. It is biased by the blended
and censored windows (§11). Here it is fitted to every detection.

In [ ]:
# section 4: the design matrix of the detection-set fit (the probe), on every detection
# dyn.motion_design: X of Eq. 15, one row per detection, the columns sin, cos, t sin, t cos
X = dyn.motion_design(det_a["psi"], det_a["t"])
print("design matrix, first three rows [sin, cos, t sin, t cos]:")
print(X[:3].round(3))

In [ ]:
# dyn.fit_motion: the weighted least-squares solve of Eq. 15 (Eq. 16 with a parallax column); returns theta, its
# covariance, and the detections moved to the reference epoch (not used here)
probe_all, cov_all, _ = dyn.fit_motion(det_a["psi"], det_a["t"], det_a["eta"], det_a["sigma"])
# the same solve written out: the normal equations of Eq. 15. W is the diagonal matrix of the weights 1/sigma^2
W = np.diag(1.0 / det_a["sigma"] ** 2)
theta_hand = np.linalg.solve(X.T @ W @ X, X.T @ W @ det_a["eta"])
assert np.allclose(theta_hand, probe_all, rtol=1e-10) and np.allclose(np.linalg.inv(X.T @ W @ X), cov_all, rtol=1e-10)

In [ ]:
# truth: the probe's four parameters and their errors, beside the injected values
NAMES_4 = ("a0 [mas]", "d0 [mas]", "mu_a [mas/yr]", "mu_d [mas/yr]")   # the four parameters, in theta's order
probe_all_err = np.sqrt(np.diag(cov_all))                  # the 1-sigma errors: the root of the covariance's diagonal
for name, value, error, truth in zip(NAMES_4, probe_all, probe_all_err, TRUTH_A):
    print(f"  {name:14s} {value:8.2f} +/- {error:.2f}   truth {truth:8.2f}")

## 5. The seed: the map detects, the sinogram refines (Sect. 4.2, Eq. 17)

Fig. 2's three steps, for a source that moves:

1. **Detect.** The map of the detections as measured; its strongest peak outside 30 mas of the primary. The lines
   of a moving source do not cross at one point, so the peak is broad and is not the position at the reference
   epoch.
2. **Select.** The detections whose line passes within 30 mas of the peak: a wide circle, to hold the lines the
   motion has carried away from it.
3. **Refine.** Fit the sinogram with motion (Eq. 15) to the selected detections, by the Huber estimator
   (`dyn.fit_motion_robust`), which a few lines of another source inside the circle pull less than they pull least
   squares.
4. **Clean.** Test every detection against the fitted track at its own epoch, $|\hat\eta_i - \eta(\psi_i, t_i)|
   \le \max(3\sigma_{\eta,i}, w)$ with $w$ the peak's width; refit Eq. 15 by least squares to those that pass;
   repeat until the set that passes is the set last fitted. That fit is the **seed** of step 7.

**Eq. 17 and the map.** With a fitted motion, each detection is taken back to the reference epoch,

$$\hat\eta^{\rm dyn}_i = \hat\eta_i - \left(\mu_{\alpha^*}\tau_i\sin\psi_i + \mu_\delta\tau_i\cos\psi_i + \Delta\varpi\,f_{\varpi,i}\right), \qquad (17)$$

and the map of the compensated detections (dynamical backprojection, Eq. 13) has its crossing at the
reference-epoch position. The residual tested in step 4 is the distance of that compensated line from the fitted
position, so step 4 is the test "the line passes through the compensated map's peak" without building the map.
The figure's bottom row draws the two maps.

In [ ]:
# section 5: the seed written out as one function (detect, select, refine, clean)
SELECT_RADIUS_MAS = 30.0                   # the circle of step 2 about the map's peak [mas]
ORIGIN_ZONE_MAS = 30.0                     # step 1: no peak is taken within this distance of the primary [mas]
K_SIGMA = 3.0                              # step 4: a detection may miss the track by this many of its error bars
MAX_PASSES = 5                             # step 4: the cap on the number of passes of the cleaning


def clean_by_hand(det, parallax, max_pass=MAX_PASSES, verbose=False):
    """The seed of section 5: the map detects, a circle selects, the sinogram fit refines and cleans.

    Inputs
        det        the table of detections (from reduce_windows)
        parallax   True: fit a parallax difference as a fifth parameter; False: four parameters
        max_pass   the cap on the passes of step 4
        verbose    True: print the peak and each pass
    Returns four things
        theta    the last least-squares fit, the seed of section 7: (alpha_0, delta_0 [mas], mu_alpha*, mu_delta
                 [mas/yr][, d_parallax [mas]])
        cov      its covariance
        keep     a mask over the detections: True for those the last fit was fitted to
        passes   the number of passes of step 4 that ran"""
    # step 0, the detections' columns, and X, the design matrix of Eq. 15 (with the f_parallax column when
    # parallax is True): X @ theta is the fitted track at every detection's angle and epoch
    psi_det = det["psi"]
    t_det = det["t"]
    eta_det = det["eta"]
    sigma_det = det["sigma"]
    if parallax:
        fpar_det = det["fpar"]
    else:
        fpar_det = None
    X = dyn.motion_design(psi_det, t_det, fpar_det, parallax)

    # step 1, detect. bp.map_peak: the highest peak of the detections' map outside the origin zone, (dra*, ddec)
    # [mas], and the map's width there [mas]; it also returns the map and its axis (dropped here, the two _)
    peak, width, _, _ = bp.map_peak(psi_det, eta_det, sigma_det, origin_zone=ORIGIN_ZONE_MAS)

    # step 2, select. bp.peak_consistency returns a mask (dropped here, the name _) and each line's distance from
    # the peak [mas]; the detections whose line passes within SELECT_RADIUS_MAS of the peak are selected
    _, distance_from_peak = bp.peak_consistency(psi_det, eta_det, sigma_det, peak, width, k_sigma=K_SIGMA)
    selected = distance_from_peak <= SELECT_RADIUS_MAS

    # step 3, refine. dyn.fit_motion_robust: the Huber fit of Eq. 15 to the selected detections ([selected] keeps
    # them). It returns theta, a covariance, the weights and the compensated detections; [0] keeps theta
    if fpar_det is None:
        fpar_selected = None
    else:
        fpar_selected = fpar_det[selected]
    robust_fit = dyn.fit_motion_robust(psi_det[selected], t_det[selected], eta_det[selected], sigma_det[selected],
                                       fpar_selected, parallax=parallax)
    theta = robust_fit[0]
    if verbose:
        print(f"  peak ({peak[0]:.0f}, {peak[1]:.0f}) mas, width {width:.1f} mas, "
              f"{selected.sum()} of {selected.size} lines within {SELECT_RADIUS_MAS:.0f} mas")

    # step 4, clean. Each pass tests every detection against the fitted track at its own epoch, then refits by
    # least squares to those that pass; it stops when the set that passes is the set last fitted
    keep = None
    cov = None
    for n_pass in range(1, max_pass + 1):
        # a detection passes when it is within max(K_SIGMA sigma, width) of the track
        passed = np.abs(eta_det - X @ theta) <= np.maximum(K_SIGMA * sigma_det, width)
        if verbose:
            print(f"  pass {n_pass}: kept {passed.sum()} of {passed.size}")
        if keep is not None and np.array_equal(passed, keep):
            return theta, cov, keep, n_pass
        keep = passed
        if fpar_det is None:
            fpar_kept = None
        else:
            fpar_kept = fpar_det[keep]
        # dyn.fit_motion: least squares on the kept detections; its third return (the compensated detections) is
        # dropped
        theta, cov, _ = dyn.fit_motion(psi_det[keep], t_det[keep], eta_det[keep], sigma_det[keep],
                                       fpar_kept, parallax=parallax)
    return theta, cov, keep, max_pass

In [ ]:
# the seed of case (a): the four steps, printed as they run
seed_a, seed_cov_a, keep_a, passes_a = clean_by_hand(det_a, parallax=False, verbose=True)
# Eq. 17 by hand with the seed's motion, against dyn.to_reference_epoch
mu_a_seed = seed_a[2]                      # the third entry of the seed: mu_alpha* [mas/yr]
mu_d_seed = seed_a[3]                      # the fourth entry: mu_delta [mas/yr]
eta_dyn = det_a["eta"] - (mu_a_seed * det_a["t"] * np.sin(det_a["psi"])
                          + mu_d_seed * det_a["t"] * np.cos(det_a["psi"]))
eta_dyn_package = dyn.to_reference_epoch(seed_a, det_a["psi"], det_a["t"], det_a["eta"])
assert np.allclose(eta_dyn, eta_dyn_package, rtol=0, atol=1e-9)

In [ ]:
# truth: the seed's four parameters and their errors, beside the injected values
seed_err_a = np.sqrt(np.diag(seed_cov_a))                  # the 1-sigma errors: the root of the covariance's diagonal
for name, value, error, truth in zip(NAMES_4, seed_a, seed_err_a, TRUTH_A):
    print(f"  {name:14s} {value:8.2f} +/- {error:.2f}   truth {truth:8.2f}")

In [ ]:
# the convenience: the same steps in one call (dyn.clean_detection_set; static=True gives Fig. 2's form)
# it returns theta, its covariance (dropped here, the name _), the kept set, and a dictionary on how it ran
theta_one_call, _, keep_one_call, info_one_call = dyn.clean_detection_set(det_a["psi"], det_a["t"], det_a["eta"],
                                                                         det_a["sigma"])
assert np.array_equal(theta_one_call, seed_a) and np.array_equal(keep_one_call, keep_a)
assert info_one_call["iterations"] == passes_a and info_one_call["converged"]
print("dyn.clean_detection_set reproduces the seed and the kept set exactly")

## 6. The window-level likelihood (Sect. 4.3, Eq. 18)

Eq. 18 with θ = (α₀, δ₀, μα\*, μδ[, Δϖ]): θ fixes the companion's η in every window at its epoch
(Eq. 16); the model of all windows is their stacked profiles times the fluxes; the fluxes,
constant over the span, follow from one non-negative solve, leaving the profiled likelihood of θ.
The primary is held at the origin (exact here).

In [ ]:
# section 6: Eq. 18 for the moving scene, at the truth -- the sources' along-scan positions and the stacked
# model
print("eta of each source, first three windows [mas]:")
# joint.scene_positions_al: Eq. 16 for every source at a theta; psi[:3] and t[:3] keep the first three windows
print(joint.scene_positions_al(TRUTH_A, psi[:3], t[:3]).round(1))
# joint.stacked_design: the matrix A of the model A f. One row per sample of all the windows, one column per
# source (the primary, the companion): each source's profile in every window for one unit of flux
A_stack = joint.stacked_design(TRUTH_A, psi, inst, t)
print(f"stacked model: {A_stack.shape[0]} samples x {A_stack.shape[1]} sources; the model is A f")

In [ ]:
# joint.loglike_profiled: ln L of Eq. 18 at theta, the fluxes set by a non-negative solve (returned on request);
# joint.loglike_windows: the same ln L at given fluxes, so the two agree at the profiled fluxes
# with return_fluxes=True it returns ln L (dropped here, the name _) and the fluxes
_, flux_at_truth = joint.loglike_profiled(TRUTH_A, windows_a, psi, inst, sigma_noise, t, return_fluxes=True)
print(f"profiled fluxes at the truth    {flux_at_truth.round(1)}   (injected {scene_a.fluxes})")
lnl_at_given_fluxes = joint.loglike_windows(TRUTH_A, flux_at_truth, windows_a, psi, inst, sigma_noise, t)
lnl_profiled = joint.loglike_profiled(TRUTH_A, windows_a, psi, inst, sigma_noise, t)
assert np.isclose(lnl_at_given_fluxes, lnl_profiled, rtol=1e-12)

In [ ]:
# the profiled ln L of Eq. 18 at the truth and at the seed
for label, theta in (("truth", TRUTH_A), ("seed", seed_a)):
    lnl = joint.loglike_profiled(theta, windows_a, psi, inst, sigma_noise, t)
    print(f"ln L profiled at the {label:6s}     "
          f"{lnl:.3f} nats")

## 7. The fit (Sect. 4.3)

The maximum of the profiled likelihood from the seed (restarted bounded Nelder–Mead); then the
fluxes, the Fisher matrix and the covariance at the solution.

In [ ]:
# section 7: the maximum of the profiled likelihood from the seed, written out as one function
def fit_by_hand(windows, sigma_noise, seed, parallax):
    """The window-level fit of the moving scene: the maximum of the profiled likelihood of Eq. 18, from a seed.

    Inputs
        windows       the list of windows, one array of samples per transit
        sigma_noise   the noise sigma of every sample [flux per sample]
        seed          the starting theta: (alpha_0, delta_0 [mas], mu_alpha*, mu_delta [mas/yr][, d_parallax [mas]])
        parallax      True: the seed has the parallax difference as its fifth entry
    Returns four things
        theta         the fitted parameters, in the seed's order and units
        sigma_theta   their 1-sigma errors
        flux          the fitted fluxes [flux per sample]: the primary's, then the companion's
        cov           the covariance of the parameters and the fluxes together (None when ill-conditioned)"""
    # step 1, the parallax factors: every transit's when a parallax difference is fitted, None otherwise
    if parallax:
        fpar_fit = fpar
    else:
        fpar_fit = None

    # step 2, the bounds: the two offsets within +/- joint.THETA_BOUND [mas], the motion (and the parallax
    # difference) free. A list times a number repeats the list
    n_par = seed.size
    offset_bounds = [(-joint.THETA_BOUND, joint.THETA_BOUND)] * 2
    motion_bounds = [(None, None)] * (n_par - 2)
    bounds = offset_bounds + motion_bounds

    # step 3, the search. The optimizer minimises, so it is given -ln L. opt.minimize_restarts: a bounded
    # Nelder-Mead search, restarted from its answer until it no longer improves. It returns the result, the number
    # of restarts (dropped here, the name _) and whether it converged
    def negative_loglike(theta_trial):
        return -joint.loglike_profiled(theta_trial, windows, psi, inst, sigma_noise, t, fpar_fit)

    result, _, converged = opt.minimize_restarts(negative_loglike, seed, bounds=bounds, xatol=1e-4,
                                                 maxiter_first=8000, maxiter_restart=32000)
    assert converged, "the restart loop ran out of restarts while still improving"
    theta = np.asarray(result.x)

    # step 4, the fluxes at the solution (the ln L that comes with them is dropped, the name _)
    _, flux = joint.loglike_profiled(theta, windows, psi, inst, sigma_noise, t, fpar_fit, return_fluxes=True)

    # step 5, the errors. joint.joint_fisher: the Fisher matrix of theta and the fluxes at the solution;
    # joint.joint_covariance: its inverse and the 1-sigma errors of the first n_par entries, the position and motion
    fisher = joint.joint_fisher(theta, flux, psi, inst, sigma_noise, t, fpar_fit)
    cov, sigma_theta = joint.joint_covariance(fisher, n_par)
    return theta, sigma_theta, flux, cov

In [ ]:
# the fit of case (a), from the seed
theta_a, sigma_theta_a, flux_a, cov_a = fit_by_hand(windows_a, sigma_noise, seed_a, parallax=False)

In [ ]:
# truth: the fit's four parameters and their errors beside the injected values, and the flux ratio
# (companion over primary: flux_a[1] / flux_a[0])
for name, value, error, truth in zip(NAMES_4, theta_a, sigma_theta_a, TRUTH_A):
    print(f"  {name:14s} {value:8.2f} +/- {error:.2f}   truth {truth:8.2f}   ({(value - truth) / error:+.1f} sigma)")
print(f"  q = {flux_a[1] / flux_a[0]:.4f}   (truth {Q})")

In [ ]:
# the convenience
# joint.fit_joint: the same fit in one call; tau is the epochs. It returns a dictionary with the entries "theta",
# "sigma_theta", "flux" and "cov" (among others)
fit_one_call = joint.fit_joint(windows_a, psi, seed_a, inst, sigma_noise, tau=t)
assert np.array_equal(fit_one_call["theta"], theta_a) and np.array_equal(fit_one_call["sigma_theta"], sigma_theta_a)
assert np.array_equal(fit_one_call["flux"], flux_a) and np.array_equal(fit_one_call["cov"], cov_a)
print("joint.fit_joint reproduces the solution and its covariance exactly")

## 8. The compensated sinogram (Fig. 3, top row)

Every detection taken back to the reference epoch with the **fit's** motion (Eq. 17), as the figure
draws it: kept ones land on the fitted track.

In [ ]:
# section 8: every detection moved to the reference epoch with the fit's motion; the kept ones should land on
# the fitted track, the rejected ones off it
eta_compensated_a = dyn.to_reference_epoch(theta_a, det_a["psi"], det_a["t"], det_a["eta"])
# the fitted track at the reference epoch: Eq. 9 at the fitted offset (theta_a[0], theta_a[1]) = (alpha_0, delta_0)
eta_track_a = ins.project_al(theta_a[0], theta_a[1], det_a["psi"])
residual = eta_compensated_a - eta_track_a
# keep_a is the mask of the cleaning (section 5): residual[keep_a] the kept detections, residual[~keep_a] the
# rejected ones
print(f"kept:     |residual from the t0 track| median {np.median(np.abs(residual[keep_a])):.1f} mas")
print(f"rejected: residuals {residual[~keep_a].round(1)} mas, "
      f"{(residual[~keep_a] / det_a['sigma'][~keep_a]).round(1)} error bars")

## 9. The three runs (Sect. 4.5)

Steps 2–8 composed into one function, run on (a), on (a) with Δϖ as a fifth parameter (the null
test, zero expected), and on (b). The table: the probe on every detection, the cleaned probe, the
window-level fit, each with its pull against the injected value.

In [ ]:
# section 9: steps 2-8 as one function (chain), which returns what the table and the figure need
def chain(scene, parallax):
    """Every stage of this notebook (sections 2 to 8) for one scene.

    Inputs
        scene      a sim.Scene (scene_a or scene_b)
        parallax   True: every fit has the parallax difference as a fifth parameter
    Returns a dictionary, the record of one run
        "psi", "t", "eta", "sigma", "fpar", "window"   the table of detections (see reduce_windows)
        "eta_compensated"   the detections moved to the reference epoch with the window-level fit's motion [mas]
        "keep", "passes"    the mask of the detections the cleaning kept, and its number of passes
        "probe_all", "probe_all_err"   the detection-set fit to every detection, and its 1-sigma errors
        "probe", "probe_err"           the detection-set fit to the kept detections (the seed), and its errors
        "theta", "err", "flux"         the window-level fit, its 1-sigma errors and its fluxes
    Every fit is (alpha_0, delta_0 [mas], mu_alpha*, mu_delta [mas/yr][, d_parallax [mas]])."""
    # step 1, sections 2 and 3: the windows (the true positions are dropped, the name _) and their detections
    windows, sigma_noise, _ = render(scene)
    det = reduce_windows(windows, sigma_noise)
    if parallax:
        fpar_det = det["fpar"]
    else:
        fpar_det = None

    # step 2, section 4: the detection-set fit to every detection (its third return is dropped)
    probe_all, cov_all, _ = dyn.fit_motion(det["psi"], det["t"], det["eta"], det["sigma"], fpar_det,
                                           parallax=parallax)

    # step 3, section 5: the seed
    probe, cov_probe, keep, passes = clean_by_hand(det, parallax)

    # step 4, section 7: the window-level fit from the seed (its covariance is dropped, the name _)
    theta, err, flux, _ = fit_by_hand(windows, sigma_noise, probe, parallax)

    # step 5, section 8: the detections at the reference epoch, with the window-level fit's motion
    eta_compensated = dyn.to_reference_epoch(theta, det["psi"], det["t"], det["eta"], fpar_det, parallax)

    # step 6, the record; the 1-sigma errors of the two detection-set fits are the roots of their covariances'
    # diagonals
    return dict(psi=det["psi"], t=det["t"], eta=det["eta"], sigma=det["sigma"], fpar=det["fpar"],
                window=det["window"], eta_compensated=eta_compensated, keep=keep, passes=passes,
                probe_all=probe_all, probe_all_err=np.sqrt(np.diag(cov_all)),
                probe=probe, probe_err=np.sqrt(np.diag(cov_probe)),
                theta=theta, err=err, flux=flux)

In [ ]:
# the three runs: (a), which must reproduce the steps above exactly; (a) with a parallax difference; (b)
run_a = chain(scene_a, parallax=False)
assert np.array_equal(run_a["theta"], theta_a) and np.array_equal(run_a["keep"], keep_a)
assert np.array_equal(run_a["eta_compensated"], eta_compensated_a)
run_a_null = chain(scene_a, parallax=True)   # the null test: case (a) fitted with a parallax difference
run_b = chain(scene_b, parallax=True)        # case (b): the chance alignment

In [ ]:
# report: the function that prints one case's table
def report(tag, run, truth, names):
    """Print one case's table: the probe on every detection, the cleaned probe and the window-level fit, each
    with its pull against the truth.

    Inputs
        tag     the case's name, printed first
        run     the record of one run (from chain)
        truth   the injected parameters, in the fits' order and units
        names   the printed name of each parameter
    Returns nothing."""
    # step 1, the head: how many detections, how many the cleaning kept
    n_detections = run["psi"].size
    n_kept = int(run["keep"].sum())
    print(f"{tag}: {n_detections} detections, {n_kept} kept, {n_detections - n_kept} rejected by pooling "
          f"in {run['passes']} passes; max sigma_eta {run['sigma'].max():.0f} mas")
    print(f"    {'':>10s}  {'probe, all detections':>26s}  "
          f"{'probe, cleaned':>26s}  {'window-level fit':>26s}  {'truth':>8s}")

    # step 2, one line per parameter (i counts the parameters): for each of the three fits its value, its error
    # and its pull, (value - truth) / error, in units of sigma (the printed "s")
    for i, name in enumerate(names):
        columns = []
        for estimate, error in ((run["probe_all"], run["probe_all_err"]),
                                (run["probe"], run["probe_err"]), (run["theta"], run["err"])):
            columns.append(f"{estimate[i]:8.2f} +/- {error[i]:4.2f} ({(estimate[i] - truth[i]) / error[i]:+4.1f}s)")
        print(f"    {name:>10s}  " + "  ".join(columns) + f"  {truth[i]:8.2f}")

In [ ]:
# truth: the tables of (a) and (b), and the fitted parallax difference of the null test and of (b)
# [4] keeps the fifth parameter of a fit with parallax, the parallax difference, and its error
dpi_a = run_a_null["theta"][4]
err_dpi_a = run_a_null["err"][4]
dpi_b = run_b["theta"][4]
err_dpi_b = run_b["err"][4]
report("(a) bound companion", run_a, TRUTH_A, ["a0 [mas]", "d0 [mas]", "mu_a", "mu_d"])
print(f"    null test ({int(run_a_null['keep'].sum())} of {run_a_null['psi'].size} kept): d_parallax "
      f"{dpi_a:+.2f} +/- {err_dpi_a:.2f} mas ({dpi_a / err_dpi_a:+.1f} sigma); none injected")
report("(b) interloper", run_b, TRUTH_B, ["a0 [mas]", "d0 [mas]", "mu_a", "mu_d", "d_parallax"])
print(f"    d_parallax {dpi_b:+.2f} +/- {err_dpi_b:.2f} mas ({dpi_b / err_dpi_b:+.1f} sigma "
      f"from zero); {DPI_INTERLOPER:.2f} injected -> not a bound companion")

In [ ]:
# the checks stated above
for run in (run_a, run_a_null, run_b):
    assert run["passes"] < MAX_PASSES, "the cleaning must settle within its cap"
    assert 0 < run["keep"].sum() < run["psi"].size, "some, not all, detections kept"
assert abs(dpi_a / err_dpi_a) < 3.0, "the null test should be consistent with zero"
assert dpi_b < 0 and abs(dpi_b / err_dpi_b) > 3.0, "the background interloper"
# every parameter of the window-level fit within 4 sigma of the truth, of the cleaned probe within 3 sigma
for run, truth in ((run_a, TRUTH_A), (run_b, TRUTH_B)):
    assert (np.abs(run["theta"] - truth) / run["err"]).max() < 4.0
    assert (np.abs(run["probe"] - truth) / run["probe_err"]).max() < 3.0

# Sect. 4.5: (3.0 +/- 0.6, 8.9 +/- 0.4) mas/yr; Delta varpi -0.1 +/- 1.0 (null test) and -7.4 +/- 1.0 mas
# [2:4] keeps the two proper-motion entries (mu_a, mu_d); flux[1] / flux[0] is the companion over the primary
assert np.array_equal(np.round(run_a["theta"][2:4], 2), [3.05, 8.89])
assert np.array_equal(np.round(run_a["err"][2:4], 2), [0.56, 0.37])
assert round(run_a["flux"][1] / run_a["flux"][0], 3) == 0.302
assert (round(dpi_a, 2), round(err_dpi_a, 2)) == (-0.07, 0.96)
assert (round(dpi_b, 2), round(err_dpi_b, 2)) == (-7.45, 0.95)
# the values as Sect. 4.5 quotes them, to one decimal: -0.1 +/- 1.0 and -7.4 +/- 1.0 mas (-7.45 above is -7.4497)
assert (f"{dpi_a:.1f}", f"{err_dpi_a:.1f}") == ("-0.1", "1.0")
assert (f"{dpi_b:.1f}", f"{err_dpi_b:.1f}") == ("-7.4", "1.0")
assert round(dpi_b / err_dpi_b, 1) == -7.8
print("checks passed")

### Where the scans go

Scans leave the detection set at two places: **at the window** (censored, one source; they still
enter the window-level fit) and **at the cleaning** (the line misses the compensated peak by more than
3σ: the crossed markers). The window-level fit reaches the same optimum from the cleaned and the
uncleaned seed.

In [ ]:
# where the scans go: per case, the censored windows, the rejected detections, and the window-level fit from
# the uncleaned seed (it must reach the same optimum)
for tag, run, scene, parallax in (("(a) bound", run_a, scene_a, False), ("(b) chance alignment", run_b, scene_b, True)):
    # the same windows again (the same seed); the true positions are dropped, the name _
    windows_again, sigma_noise_again, _ = render(scene)
    # the window-level fit from the probe on every detection; [0] keeps theta, the first of fit_by_hand's four
    # returns
    fit_from_uncleaned = fit_by_hand(windows_again, sigma_noise_again, run["probe_all"], parallax)
    theta_uncleaned = fit_from_uncleaned[0]
    # np.unique keeps each window's number once: the windows that gave a detection
    n_censored = psi.size - np.unique(run["window"]).size
    rejected = ~run["keep"]                                # the mask of the detections the cleaning rejected
    print(f"{tag}: {n_censored} of {psi.size} scans censored; "
          f"{int(rejected.sum())} detections rejected in {run['passes']} passes; "
          f"the fit from the uncleaned seed differs by at most "
          f"{np.abs(theta_uncleaned - run['theta']).max():.1e} (mas, mas/yr)")
    assert np.abs(theta_uncleaned - run["theta"]).max() < 1e-3
    # truth: each rejected detection against the companion's true eta at its epoch; along_scan(...)[1]
    # keeps the companion (index 0 is the primary). i is the detection's place in the table
    for i in np.flatnonzero(rejected):
        true_eta = scene.along_scan(run["psi"][i], run["t"][i], run["fpar"][i])[1]
        print(f"    psi {np.rad2deg(run['psi'][i]) % 360:6.1f} deg   eta {run['eta'][i]:6.1f} mas   "
              f"sigma {run['sigma'][i]:5.1f} mas   "
              f"{(run['eta'][i] - true_eta) / run['sigma'][i]:+.1f} sigma off the true track")
# per case: (detections, censored windows, rejected detections)
assert (run_a["psi"].size, psi.size - np.unique(run_a["window"]).size, int((~run_a["keep"]).sum())) == (34, 13, 1)
assert (run_b["psi"].size, psi.size - np.unique(run_b["window"]).size, int((~run_b["keep"]).sum())) == (32, 15, 2)

## 10. The figure (Fig. 3)

Top: the sinograms of (a) and (b); kept detections raw (open circle) and compensated (black point),
rejected ones crossed, the fitted reference-epoch track. Bottom: the map of the raw detections and of
the kept, compensated ones, zoomed on the companion. Plotting only.

In [ ]:
# Figure 3: the compensated sinograms (top) and the maps before and after the compensation (bottom). Plotting only
psi_grid = np.linspace(0, 360, 400)        # the scan angles of the drawn track [deg]


def sinogram_panel(ax, psi_det, eta_raw, eta_compensated, sigma_det, keep, alpha0_fit, delta0_fit, label):
    """Kept detections raw (open) -> stem -> compensated (black), rejected ones
    crossed at their compensated value; the fitted reference-epoch track.

    Inputs: the axes; the detections' scan angles [rad], their eta as measured and at the reference epoch [mas],
    their errors [mas] and the mask of the kept ones; the fitted offset at the reference epoch [mas]; the panel's
    label."""
    psi_deg = np.rad2deg(psi_det) % 360    # the scan angles in degrees, 0 to 360
    rejected = ~keep                       # the mask of the detections the cleaning rejected
    # one gray stem per kept detection, from its raw value (y0) to its compensated value (y1) at its angle xi
    for xi, y0, y1 in zip(psi_deg[keep], eta_raw[keep], eta_compensated[keep]):
        ax.plot([xi, xi], [y0, y1], color="0.7", lw=0.6, zorder=1)
    ax.errorbar(psi_deg[keep], eta_compensated[keep], yerr=sigma_det[keep], fmt="none",
                ecolor="k", elinewidth=0.3, capsize=0.0, zorder=1.8)
    ax.scatter(psi_deg[keep], eta_raw[keep], s=50, facecolor="white",
               edgecolor="0.55", lw=0.7, zorder=2, label="raw")
    ax.scatter(psi_deg[keep], eta_compensated[keep], s=12, color="k", zorder=4,
               label="compensated")
    ax.scatter(psi_deg[rejected], eta_compensated[rejected], s=60, marker="x", color="k", lw=1.1,
               zorder=4, label="rejected by pooling")
    # the fitted track at the reference epoch: Eq. 9 at the fitted offset
    ax.plot(psi_grid, alpha0_fit * np.sin(np.deg2rad(psi_grid)) + delta0_fit * np.cos(np.deg2rad(psi_grid)),
            "-", color="k", lw=1.0, zorder=3, label=r"fitted $t_0$")
    ax.legend(loc="lower left", ncol=2, fontsize=7.0, framealpha=0.9,
              handlelength=1.4, columnspacing=1.0, borderpad=0.4)
    ax.set_xlim(0, 360)
    ax.set_xticks([0, 90, 180, 270, 360])
    ax.set_xlabel(r"scan angle $\psi$ [deg]")
    ax.set_ylabel(r"detected $\hat\eta$ [mas]")
    ax.tick_params(direction="in")
    ax.text(0.025, 0.96, label, transform=ax.transAxes, va="top", ha="left",
            fontsize=9, fontweight="bold", bbox=dict(boxstyle="round", fc="white", ec="none",
                                                     alpha=0.9))


def map_panels(ax_raw, ax_cor, psi_det, eta_raw, eta_compensated, sigma_det, keep, alpha0_true, delta0_true,
               half_zoom=45.0):
    """Raw and motion-compensated maps, zoomed to +/- half_zoom mas about the true crossing.

    Inputs: the two axes; the detections (as in sinogram_panel); the companion's true offset at the reference
    epoch [mas]; the half-width of the zoom [mas]."""
    # left: every detection as measured (np.ones_like(keep) is a mask that is True everywhere); right: the kept
    # detections at the reference epoch
    for ax, used, eta_used, is_compensated in [(ax_raw, np.ones_like(keep), eta_raw, False),
                                               (ax_cor, keep, eta_compensated, True)]:
        # bp.backprojection_map: the triangulation map of Eq. 13, its axis [mas] and its extent (dropped, the _)
        sky_map, axis, _ = bp.backprojection_map(psi_det[used], eta_used[used], sigma_det[used],
                                                 half_size=300.0, step=1.0)
        # the pixels of the zoom: in_alpha along alpha*, in_delta along delta
        in_alpha = np.abs(axis - alpha0_true) < half_zoom
        in_delta = np.abs(axis - delta0_true) < half_zoom
        # np.ix_(in_delta, in_alpha) keeps the rows in in_delta and the columns in in_alpha
        ax.imshow(sky_map[np.ix_(in_delta, in_alpha)], origin="lower", aspect="equal", cmap="gray_r",
                  extent=(axis[in_alpha][-1], axis[in_alpha][0], axis[in_delta][0], axis[in_delta][-1]))
        ax.plot(alpha0_true, delta0_true, "*", color="white", mec="k", ms=13)
        ax.set_xlim(alpha0_true + half_zoom, alpha0_true - half_zoom)      # east to the left
        ax.set_ylim(delta0_true - half_zoom, delta0_true + half_zoom)
        ax.set_anchor("N")
        ax.set_xlabel(r"$\Delta\alpha^{*}$ [mas] (E$\leftarrow$)")
        ax.tick_params(direction="in", color="0.5")
        if ax is ax_raw:
            ax.set_ylabel(r"$\Delta\delta$ [mas]")
        ax.text(0.05, 0.95, "corrected" if is_compensated else "raw", transform=ax.transAxes,
                va="top", ha="left", fontsize=8.5,
                bbox=dict(boxstyle="round", fc="white", ec="none", alpha=0.85))


fig = plt.figure(figsize=(15.0, 7))
gs = fig.add_gridspec(2, 4, height_ratios=[1., 1.], hspace=0.3)
ax_sa = fig.add_subplot(gs[0, 0:2])        # top left: the sinogram of case (a)
ax_sb = fig.add_subplot(gs[0, 2:4])        # top right: the sinogram of case (b)
# theta[0], theta[1]: the fitted offset at the reference epoch (alpha_0, delta_0)
sinogram_panel(ax_sa, run_a["psi"], run_a["eta"], run_a["eta_compensated"], run_a["sigma"], run_a["keep"],
               run_a["theta"][0], run_a["theta"][1], "(a) bound")
sinogram_panel(ax_sb, run_b["psi"], run_b["eta"], run_b["eta_compensated"], run_b["sigma"], run_b["keep"],
               run_b["theta"][0], run_b["theta"][1], "(b) chance alignment")
ax_sb.set_ylabel("")                       # (b) shares (a)'s eta axis
map_panels(fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]),
           run_a["psi"], run_a["eta"], run_a["eta_compensated"], run_a["sigma"], run_a["keep"],
           alpha0_true, delta0_true)
ax_mb = fig.add_subplot(gs[1, 2])
map_panels(ax_mb, fig.add_subplot(gs[1, 3]),
           run_b["psi"], run_b["eta"], run_b["eta_compensated"], run_b["sigma"], run_b["keep"],
           alpha0_true, delta0_true)
ax_mb.set_ylabel("")                       # case (b) shares case (a)'s delta axis
fig.tight_layout(w_pad=2.0, h_pad=1.0)

In [ ]:
# write the PDF, only when SAVE_PDF is True
if SAVE_PDF:
    Path("pdf").mkdir(exist_ok=True)
    fig.savefig(Path("pdf") / "fig3_moving_companion.pdf", bbox_inches="tight")
    print("wrote pdf/fig3_moving_companion.pdf")
else:
    print("SAVE_PDF is False: the PDF is not written")

## 11. Over 300 draws: the detection-set fit is biased, the window-level fit is not (Sect. 4.5)

One draw cannot show a bias. `auxil/sim_coverage_fig3.ipynb` stores, for 300 draws of each case, the
cleaned probe and the window-level fit started from it. Per parameter: the bias (mean of fitted minus
true, with its standard error), the scatter, and the pull width (1 when the quoted errors are right).

In [ ]:
# section 11: read the 300 draws per case stored by auxil/sim_coverage_fig3, and check that the store's
# scenes are this notebook's
STORE = Path("stored") / "fig3_coverage.npz"     # written by auxil/sim_coverage_fig3.ipynb
# store: a table from an array's name in the file to the array; every array has the case as its first axis
# (0 bound, 1 null test, 2 interloper)
store = {}
with np.load(STORE) as stored_file:
    for array_name in stored_file.files:
        store[array_name] = stored_file[array_name]
# the store's truth has one row per case and five columns: [0, :4] keeps case (a) and omits its parallax
# column; [2] is case (b)
assert np.array_equal(store["truth"][0, :4], TRUTH_A) and np.array_equal(store["truth"][2], TRUTH_B)
# FITS: the two fits of the store, each (its printed name, the prefix of its arrays' names in the store): the
# detection-set fit is in "probe_theta" and "probe_sigma", the window-level fit in "theta" and "sigma"
FITS = (("detection set", "probe_"), ("window level", ""))

In [ ]:
# offsets: the function that gives, for one case and each of the two fits, fitted minus true and its statistics
def offsets(i, n_par):
    """Fitted minus true and its statistics over the draws, for one case of the store and each of the two fits.

    Inputs
        i       the case's number in the store: 0 bound, 1 null test, 2 interloper
        n_par   the number of fitted parameters: 4, or 5 with the parallax difference
    Returns a table from a fit's name ("detection set", "window level") to a dictionary
        "difference"       fitted minus true, one row per completed draw, one column per parameter
        "bias"             its mean over the draws, per parameter
        "standard_error"   the standard error of that mean: the scatter over the root of the number of draws
        "scatter"          its standard deviation over the draws
        "pull_width"       the standard deviation of (fitted minus true) / (quoted error); 1 when the errors are
                           right
    The units are those of the parameters (mas, mas/yr); the pull width has none."""
    completed = store["ok"][i]                           # a mask over the draws: True where the chain completed
    table = {}
    for name, prefix in FITS:
        # [i] keeps the case; [completed, :n_par] keeps the completed draws and the first n_par parameters
        difference = store[prefix + "theta"][i][completed, :n_par] - store["truth"][i, :n_par]
        quoted_sigma = store[prefix + "sigma"][i][completed, :n_par]
        # mean(0) and std(0) run over the draws (axis 0), so they give one number per parameter
        table[name] = dict(difference=difference, bias=difference.mean(0),
                           standard_error=difference.std(0) / np.sqrt(len(difference)),
                           scatter=difference.std(0), pull_width=(difference / quoted_sigma).std(0))
    return table

In [ ]:
# the table: per case and parameter, each fit's bias (with its standard error), scatter and pull width
NAMES_5 = list(NAMES_4) + ["d_parallax [mas]"]             # the five parameters, in theta's order
# OFFSETS: a table from a case's name to that case's offsets(...) table
OFFSETS = {}
# i counts the cases in the store's order; n_par is the case's number of fitted parameters
for i, (tag, n_par) in enumerate((("(a) bound", 4), ("(a) null test", 5), ("(b) interloper", 5))):
    case_offsets = offsets(i, n_par)
    OFFSETS[tag] = case_offsets
    print(f"{tag}: {len(case_offsets['window level']['difference'])} draws")
    print(f"    {'':>16s}  {'detection set: bias':>21s}  {'scatter':>7s}  {'pull w':>6s}"
          f"   {'window level: bias':>20s}  {'scatter':>7s}  {'pull w':>6s}")
    for k in range(n_par):                                 # k counts the parameters
        row = []
        for name, _ in FITS:                               # the fit's name; its prefix is not needed (the _)
            stats = case_offsets[name]
            bias_in_errors = stats["bias"][k] / stats["standard_error"][k]
            row.append(f"{stats['bias'][k]:+6.2f} +/- {stats['standard_error'][k]:4.2f} ({bias_in_errors:+5.1f})"
                       f"  {stats['scatter'][k]:7.2f}  {stats['pull_width'][k]:6.2f}")
        print(f"    {NAMES_5[k]:>16s}  " + "   ".join(row))

In [ ]:
# case (a): the bias of the proper motion along and across the true motion
offsets_a = OFFSETS["(a) bound"]
# [2:4] keeps the two proper-motion entries (mu_a, mu_d); motion_direction is the unit vector along the true
# motion
true_motion = TRUTH_A[2:4]
motion_direction = true_motion / np.hypot(*true_motion)
# a @ b is the scalar product: the part of the proper-motion bias that lies along the true motion [mas/yr]
bias_along_motion = offsets_a["detection set"]["bias"][2:4] @ motion_direction
# per parameter, the scatter of the detection-set fit over that of the window-level fit
scatter_ratio = offsets_a["detection set"]["scatter"] / offsets_a["window level"]["scatter"]
print(f"(a) detection set: proper-motion bias along the true motion {bias_along_motion:+.2f} mas/yr "
      f"({bias_along_motion / np.hypot(*true_motion):+.0%} of the drift rate)")
print(f"(a) scatter, detection set over window level: {scatter_ratio.round(1)}")

In [ ]:
# Sect. 4.5: the detection-set fit is biased (in (a) every position and
# proper-motion component beyond 4 standard errors; in (b) the position beyond
# 4, the proper motion beyond 2), underestimates the drift, and quotes errors
# too small (pull widths above 1.1; in (b) the position's, its proper motion's are 1.09 and 1.10); the
# window-level fit is unbiased within 3 standard errors, its
# quoted errors are right, and it scatters 2 to 5 times less
for tag in OFFSETS:
    window_level = OFFSETS[tag]["window level"]
    assert np.all(np.abs(window_level["bias"]) < 3 * window_level["standard_error"]), tag
    detection_set = OFFSETS[tag]["detection set"]
    # the least bias, in standard errors, asked of (alpha_0, delta_0, mu_alpha*, mu_delta), and the number of
    # leading parameters whose pull width must be above 1.1
    if tag == "(b) interloper":
        least_bias = np.array([4, 4, 2, 2])
        n_wide_pulls = 2
    else:
        least_bias = 4
        n_wide_pulls = 4
    # [:4] keeps the position and the proper motion (it omits the parallax difference)
    assert np.all(np.abs(detection_set["bias"][:4]) > least_bias * detection_set["standard_error"][:4]), tag
    assert np.all(detection_set["pull_width"][:n_wide_pulls] > 1.1), tag
assert bias_along_motion < 0 and np.all((scatter_ratio > 2) & (scatter_ratio < 5))
assert np.array_equal(np.round(offsets_a["detection set"]["bias"], 2), [-0.97, -0.96, -0.41, -0.46])
assert round(bias_along_motion, 2) == -0.56
print("checks passed")

The draws of case (a), fitted minus true, position (left) and proper motion (right): gray the
detection-set fit, black the window-level fit; large markers the means, open diamonds the figure's
draw, dashed the direction of the true motion.

In [ ]:
# the draws of case (a), fitted minus true, for the two fits. Plotting only
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
with plt.rc_context(PAPER_STYLE):                                         # the paper's style, this plot only
    fig_bias, (ax_position, ax_motion) = plt.subplots(1, 2, figsize=(7.5, 3.6))
    # left: the position, entries 0 and 1 of theta (slice(0, 2)); right: the proper motion, entries 2 and 3.
    # labels: the x symbol, the y symbol and the unit
    for ax, entries, labels in ((ax_position, slice(0, 2), (r"\Delta\alpha^*_0", r"\Delta\delta_0", "mas")),
                                (ax_motion, slice(2, 4), (r"\mu_{\alpha^*}", r"\mu_\delta", "mas/yr"))):
        for name, colour, size in (("detection set", "0.65", 7), ("window level", "k", 5)):
            difference = offsets_a[name]["difference"][:, entries]     # every draw, the two entries of this panel
            ax.scatter(difference[:, 0], difference[:, 1], s=size, color=colour, lw=0, label=name)
            ax.plot(*offsets_a[name]["bias"][entries], "o", ms=9, mfc=colour, mec="white", mew=1.2, zorder=5)
        # the figure's draw: the cleaned probe and the window-level fit of section 9, minus the truth
        ax.plot(*(run_a["probe"] - TRUTH_A)[entries], "D", ms=7, mfc="none", mec="0.4", zorder=6)
        ax.plot(*(run_a["theta"] - TRUTH_A)[entries], "D", ms=7, mfc="none", mec="k", zorder=6)
        ax.axhline(0, color="0.8", lw=0.6, zorder=0)
        ax.axvline(0, color="0.8", lw=0.6, zorder=0)
        ax.set_xlabel(f"${labels[0]}$, fitted $-$ true [{labels[2]}]")
        ax.set_ylabel(f"${labels[1]}$, fitted $-$ true [{labels[2]}]")
        ax.set_aspect("equal", adjustable="datalim")
        ax.tick_params(direction="in")
    half_length = 4.0                      # half the length of the dashed line along the true motion [mas/yr]
    ax_motion.plot([-half_length * motion_direction[0], half_length * motion_direction[0]],
                   [-half_length * motion_direction[1], half_length * motion_direction[1]], "--", color="0.4", lw=0.8,
                   label="true motion")
    ax_position.legend(loc="upper left", fontsize=7.5, markerscale=2)
    ax_motion.legend(loc="upper left", fontsize=7.5, markerscale=2)
    fig_bias.tight_layout()
    plt.show()